# AdviceLine · Colab GPU run
**When do AI chatbots cross from financial guidance into advice?** This notebook does the GPU work:

1. Collects answers from **3 open models** to the **300 benchmark items** (420 answers each)
2. Labels every answer with an **LLM judge** using the rubric (`docs/rubric.md`)
3. Rewrites advice-like answers with the **guardrail** and labels them again (for the A/B test)

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells in order.

Everything is saved to Google Drive after each batch. If Colab disconnects, reconnect and run all the cells again; it carries on where it stopped.

In [ ]:
# Is a GPU attached? You should see "Tesla T4"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Save results to Google Drive
Colab deletes files when the session ends. Saving to Drive means a disconnect never loses work.
A pop-up asks you to allow access to your Drive: click **Allow**.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/AdviceLine/results'
os.makedirs(RESULTS, exist_ok=True)
print('Results will be saved to', RESULTS)

## 2. Get the AdviceLine code from GitHub

In [ ]:
%cd /content
!git clone -q https://github.com/Raida27/adviceline.git 2>/dev/null || git -C adviceline pull -q
%cd /content/adviceline
!pip install -q -e . bitsandbytes
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', torch.cuda.is_available())

**See all the code:** this copies the whole project into your Google Drive (`MyDrive/AdviceLine/code`). You can also browse it in the Files panel on the left (folder icon → `adviceline`) and double-click any file to read it.

In [ ]:
import shutil
shutil.copytree('/content/adviceline', '/content/drive/MyDrive/AdviceLine/code',
                ignore=shutil.ignore_patterns('.git', '*.egg-info', '__pycache__'), dirs_exist_ok=True)
print('All code copied to Google Drive: MyDrive/AdviceLine/code')

## 3. Quick test (about 3 minutes)
Runs 5 single-turn items and 1 conversation through the first model, so problems show up early.

In [ ]:
!python -m adviceline.generate --model hf:Qwen/Qwen3-4B-Instruct-2507 --limit 5 --results-dir /content/test_results
import pandas as pd
test = pd.read_csv('/content/test_results/responses/hf_Qwen_Qwen3-4B-Instruct-2507.csv')
for row in test.head(3).itertuples():
    print('USER:', row.user_message[:200]); print('MODEL:', row.response[:600]); print('-' * 80)

## 4. Collect answers from the three models (about 15–20 minutes each)
| Model | Made by | Size | Licence |
|---|---|---|---|
| Qwen3-4B-Instruct-2507 | Alibaba | 4B | Apache 2.0 |
| Phi-4-mini-instruct | Microsoft | 3.8B | MIT |
| SmolLM3-3B | Hugging Face | 3B | Apache 2.0 |

Each model runs in its own process, so the GPU memory is freed before the next one.

In [ ]:
MODELS = [
    'hf:Qwen/Qwen3-4B-Instruct-2507',
    'hf:microsoft/Phi-4-mini-instruct',
    'hf:HuggingFaceTB/SmolLM3-3B',
]
for m in MODELS:
    !python -m adviceline.generate --model {m} --results-dir {RESULTS}

## 5. Label every answer with the LLM judge (about 30–45 minutes)
The judge is **Qwen2.5-7B-Instruct**, loaded in 4-bit so it fits on the T4. These labels are **silver labels**: you check them against your own labels later (Cohen's kappa).

In [ ]:
JUDGE = 'hf4bit:Qwen/Qwen2.5-7B-Instruct'
!python -m adviceline.judge --judge {JUDGE} --results-dir {RESULTS}

## 6. First look at the results

In [ ]:
!python -m adviceline.analysis --results-dir {RESULTS}
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS}/summary_judge.md').read()))

## 7. Guardrail A/B test: rewrite the advice-like answers, then label them again

In [ ]:
!python -m adviceline.guardrail --rewriter hf:Qwen/Qwen3-4B-Instruct-2507 --judge {JUDGE} --results-dir {RESULTS}
!python -m adviceline.judge --judge {JUDGE} --guarded --results-dir {RESULTS}
!python -m adviceline.abtest analyse --results-dir {RESULTS}

## 8. Download the results
They also stay in Google Drive under **MyDrive/AdviceLine/results**. Put the `results` folder inside your AdviceLine folder on your Mac, then commit it to GitHub.

In [ ]:
!cd {RESULTS}/.. && zip -qr /content/adviceline_results.zip results -x 'results/models/*'
from google.colab import files
files.download('/content/adviceline_results.zip')